# Putting It All Together

```{admonition} Overview
:class: tip

**Teaching:**

**Exercises:**

**Questions:**
* Can you show me an example?

**Objectives:**
* Create a simple workflow using a cloud VM and cloud object storage.
* Understand the basics of Identity and Access Management (IAM)
* Add collaborators to a Bucket with appropriate permissions.




```

## Create a VM

Since we only create resources as we need them in the cloud.  As an exercise you will now create a VM for our analysis.  In this case will give the VM **Full** access to **Storage**. 

Using the console navigate to the "Compute Engine" service and create a new VM with the following properties.
 * Call the VM "essentials"
 * Allow the VM "Full" access to "Storage".  This can be found under "Identity and API" and then selecting  "Set access for each API" and change "Storage" to "Full". **This will allow the VM to create, read, write, and delete all storage buckets in the project"**
 * Feel free to select a larger VM by changing the machine type to something larger, for example an "e2-standard-2".
 
When you are done connect to the machine as described below.

## Connect to the VM

First login to the instance from the Cloud Shell by running the following command:
```
gcloud compute ssh essentials
```
If prompted for a zone select `n` to find it automatically.  You can see an example session below.

```
learner@cloudshell:~ (just-armor-301114)$ gcloud compute ssh essentials
Did you mean zone [us-central1-b] for instance: [essentials] (Y/n)?  n

No zone specified. Using zone [us-west2-c] for instance: [essentials].
Linux essentials 4.19.0-18-cloud-amd64 #1 SMP Debian 4.19.208-1 (2021-09-29) x86_64

The programs included with the Debian GNU/Linux system are free software;
the exact distribution terms for each program are described in the
individual files in /usr/share/doc/*/copyright.

Debian GNU/Linux comes with ABSOLUTELY NO WARRANTY, to the extent
permitted by applicable law.
Last login: Tue Nov  9 20:12:49 2021 from 34.133.99.196
learner@essentials:~$
```


## Secure the VM

We first make sure that the VM is up to date with the latest security patches by running the following commands. Note: the `sudo unattended-upgrades` command only installs security patches.

sudo apt update
sudo unattended-upgrades

## Get Example Code

We will now install `git` and use it to download the example code into your home directory.  For those of you who are unfamiliar with git, it is a way to collaboratively manage files and we will only be using it to download the example that we will be using. 

In [ ]:
sudo apt-get install --yes git

In [ ]:
cd ~

In [ ]:
git clone https://github.internet2.edu/CLASS/CLASS-Examples.git

We now change the current directory to the `landsat` directory in the `CLASS-Examples` directory that was just created by the previous git command.

In [ ]:
cd ~/CLASS-Examples/landsat/

Your prompt should now change showing the current directory as follows.
```
learner@essentials:~/CLASS-Examples/landsat$
```

In [ ]:
ls -l

## Access the bucket

First test that our tools are working and that we can access the public bucket that we will be using.

In [ ]:
gsutil ls gs://gcp-public-data-landsat/

The index file is a list of all the files in the bucket and we can use it to search and filter files.

We will get the index and uncompress the file placing it in the `data/` directory (this is ignored by git). This should take around 2 min with a `e2-medium` instance in the `us-west2` region.

In [ ]:
mkdir -v data

In [ ]:
gsutil cp gs://gcp-public-data-landsat/index.csv.gz data/

We will now uncompress the index file to make it easier to use.  This may take some time depending on the machine type you are using.

In [ ]:
gzip -df data/index.csv.gz

Again, check what happened.

In [ ]:
ls -lh data

We will now explore the data.  The `head` command simply displays the first few lines of the file.

In [ ]:
head --lines=4 data/index.csv

## Search for Data

We can see the data is well formed and what we expect.  We will now use this data to download data related to a specific point and for the Landsat 8.  The following script does a simple filter.

In [ ]:
cat search.py

We can see that the actual search data comes from the file `search.json`.  The program reads the data from the standard input and iterates over all rows in the CSV file.  It filters the results for which the image contains the pint and prints out the bucket URL for them. We are interested in all products that contain the Burr Oak Tree.

In [ ]:
cat search.json

Now lets test this on a subset of the data.

In [ ]:
head --lines=200000 data/index.csv | python3 search.py

## Download the Data

Now that we have a list of folders we are interested, we will now download them with a simple script that takes bucket addresses (URL's) and downloads them with the `gsutil` program.

In [ ]:
cat download.sh

Get the first 2 datasets

In [ ]:
head --lines=200000 data/index.csv | python3 search.py | head --lines=2 | bash download.sh

Check that the data was downloaded

In [ ]:
ls -l data

## Processing the Data

We will now use simple script to combine the color bands and export it as a PNG

In [ ]:
cat combine.py

In [ ]:
python3 combine.py
/bin/true # ignore this line used for jupyter

Oops, let's install the library (note: the output will be slightly different due to how this Lesson is built)
and while we are at it we also need to create the output directory.

In [ ]:
mkdir -v output/
sudo apt-get install python3-rasterio --yes

In [ ]:
/usr/bin/python3 combine.py

Note, the "ERROR 4" is expected.

Verify the results were created.

In [ ]:
ls -lh output

## Exporting the Results

Now that we have the output data we will create a bucket to place the results. We will first create a bucket with a reasonable set of options.

We fisrt store the bucket name in the `BUCKET` environment variable for future use.  This time we will specify a realistic set of options for a private bucket used for computation.

Options (run `gsutil mb --help` for more information):
 * `-b on` specifies uniform bucket-level access.
 * `-l $REGION` puts the data in a specific region for lower cost and lower latency.
 * `--pap enforced` turns on public access prevention to help keep data private.  
 
The uniform bucket level access (Bucket Policy Only enabled: true) puts the data access permissions (ACL) on the entire bucket, not on each object in the bucket.  This makes the permissions obvious and makes security much more predictable.
 
As usual, we must set our environment.  In this case we also set a `REGION` environment variable to indicate where in the world we want the data to be stored.


In [ ]:
BUCKET="essentials-${USER}-$(date +%F)"
REGION="us-west2"
echo "bucket: $BUCKET region: $REGION"

gsutil mb -b on -l $REGION --pap enforced "gs://$BUCKET"

Now copy the output data to the bucket. The `-r` flag recursively copies the output directory and `-m` copies the files in parallel.  Note the locations of the `-m` and `-r` switches as they apply globally and to the `cp` command respectively.

In [ ]:
gsutil -m cp -r "output" "gs://$BUCKET"

Verify that the output was uploaded.

In [ ]:
gsutil ls "gs://$BUCKET"

In [ ]:
gsutil ls -lh "gs://$BUCKET/output"

## Viewing the Results

You now can view the results by using the Google Cloud Platform Web Console and navigating to "Cloud Storage", selecting the bucket, and then the result object you wish to view (select the `.png` file).  You will need to click the "Preview" button given the large size of the image.

## Sharing Results

In order to share resources outside a project we must use the Identity Access Management service. This is a powerful tool to grant and restrict access to resources, and if not done correctly It can have serious consequences.  **Incorrect permissions can lead to exposure of sensitive data, destruction of data, and authorized use of resources that can result in huge bills.**  When in doubt, seek help.

The question "What access is really needed?" is the **Principal of Least Privilege** and is a major cornerstone of security.  We need to determine the lowest set of permissions or roles that is needed.  In our case we wish to grant the "Collaborator" the "Viewer" access to the "results bucket".  This will allow them to view, list, and download all objects in the bucket.  This illustrates that for a **resource** a **member** (identity) is granted a **permission** (think of it in this order).  Together this is called a **policy**.  Google also uses "Roles" as a collection of predefined and managed permissions.

We will now add Members to a Bucket using the Web Console.  We will use the Web Console to interactively build the policy binding by doing the following:
 * Navigation Menu -> Storage/Cloud Storage -> Browser -> Click on the Bucket Name (**Bucket Details**) -> Select the **Permissions** tab -> Click **Add** next to "Permissions" above the permissions list.
 * In the "New Principals" box add the Identity for the collaborator (another individual) as directed by the instructor.
 * Select the "Storage Object Viewer" by typing "Storage Object Viewer" in the filter and then selecting "Storage Object Viewer".  Do not use any "Legacy Storage" roles.
 * Click "Save" to save the policy.
 <!-- ![iam-storage-object-viewer](image/iam-storage-object-viewer.png) -->
 * Verify the policy is listed in the "Permissions" table on the "Bucket Details" page (you should now be on this page).

*Advanced Note:  Changes in permissions will show up on the **Home** -> **Activity** page.*

Collaborators should now be able to see the contents of the bucket by explicitly naming it as shown below.
```
student231@cloudshell:~ (t-monument-315019)$ gsutil ls gs://essentials-learner-2021-12-17
gs://essentials-learner-2021-12-17/output/

student231@cloudshell:~ (t-monument-315019)$ gsutil ls gs://essentials-learner-2021-12-17/output
gs://essentials-learner-2021-12-17/output/result-LC08_L1TP_025033_20211010_20211018_01_T1.png
gs://essentials-learner-2021-12-17/output/result-LC08_L1TP_025033_20211010_20211018_01_T1.png.aux.xml
```

## Cleanup

We will now leave the resources running in order to learn more about monitoring costs and will clean up all the resources as the end of Lesson.  **Don't forget to do this!**